# 🗄️ Consultas SQL — Ejercicios con Soluciones

Este notebook cubre SQL desde lo básico hasta consultas avanzadas con JOINs, subconsultas e índices.
Usamos **SQLite** a través de la librería estándar `sqlite3` — no requiere instalación adicional.

**Cómo usar este notebook:**
1. Lee el **enunciado** y la **explicación conceptual**.
2. Intentá escribir la consulta en la celda `# TU CÓDIGO AQUÍ`.
3. Si necesitás ayuda, desplegá la **💡 Pista**.
4. Comparate con la **✅ Solución**.

---

### 🔑 Sintaxis SQL esencial — Referencia rápida

```sql
-- Seleccionar columnas
SELECT columna1, columna2 FROM tabla WHERE condicion;

-- Unir tablas
SELECT * FROM tabla1
JOIN tabla2 ON tabla1.id = tabla2.tabla1_id;

-- Agrupar y agregar
SELECT columna, COUNT(*), SUM(valor)
FROM tabla
GROUP BY columna
HAVING COUNT(*) > 3;

-- Actualizar
UPDATE tabla SET columna = valor WHERE condicion;

-- Eliminar
DELETE FROM tabla WHERE condicion;
```

In [ ]:
# 🔧 CELDA DE SETUP — Ejecutá esto primero
import sqlite3
import pandas as pd

def ejecutar_query(conn, query, descripcion=''):
    """
    Ejecuta una consulta SELECT y muestra el resultado como DataFrame.
    Para consultas INSERT/UPDATE/DELETE hace commit automático.
    """
    query_upper = query.strip().upper()
    es_select = query_upper.startswith('SELECT') or query_upper.startswith('EXPLAIN')
    
    if descripcion:
        print(f'▶ {descripcion}')
    
    if es_select:
        df = pd.read_sql_query(query, conn)
        print(df.to_string(index=False))
        print(f'   ({len(df)} filas)\n')
        return df
    else:
        conn.execute(query)
        conn.commit()
        print('   ✅ Ejecutado correctamente\n')

print('✅ Setup listo. Función ejecutar_query() disponible.')
print('   Uso: ejecutar_query(conn, "SELECT ...", "Descripción")')

---
## 📦 Ejercicio 1 — Consulta básica con WHERE: empleados con alto salario

**Enunciado:** Diseñá una consulta SQL que muestre el nombre y salario de todos los empleados que ganen más de $50,000 al año.

### 📖 Explicación conceptual

La cláusula **WHERE** filtra filas según una condición. Es la base de cualquier consulta útil.

| Operador | Significado | Ejemplo |
|----------|-------------|---------|
| `=` | Igual | `WHERE salario = 50000` |
| `>` / `<` | Mayor / Menor | `WHERE salario > 50000` |
| `>=` / `<=` | Mayor o igual / Menor o igual | `WHERE salario >= 50000` |
| `<>` o `!=` | Distinto | `WHERE departamento <> 'IT'` |
| `BETWEEN` | En un rango | `WHERE salario BETWEEN 40000 AND 60000` |
| `LIKE` | Patrón de texto | `WHERE nombre LIKE 'A%'` |
| `IN` | En una lista | `WHERE depto IN ('Ventas', 'IT')` |
| `AND` / `OR` | Combinar condiciones | `WHERE edad > 30 AND depto = 'IT'` |

**ORDER BY** ordena los resultados:
```sql
SELECT nombre, salario FROM empleados
WHERE salario > 50000
ORDER BY salario DESC;  -- DESC = de mayor a menor
```

In [ ]:
# 🔧 CREAMOS LA BASE DE DATOS DE EMPLEADOS
conn_emp = sqlite3.connect(':memory:')  # Base de datos en memoria (se borra al cerrar)

conn_emp.executescript("""
    CREATE TABLE empleados (
        id           INTEGER PRIMARY KEY,
        nombre       TEXT    NOT NULL,
        departamento TEXT    NOT NULL,
        salario      REAL    NOT NULL,
        edad         INTEGER,
        ciudad       TEXT
    );

    INSERT INTO empleados VALUES
        (1,  'Ana García',      'Ventas',    62000, 34, 'Buenos Aires'),
        (2,  'Carlos López',    'IT',        78000, 29, 'Córdoba'),
        (3,  'María Torres',    'RRHH',      45000, 41, 'Rosario'),
        (4,  'Pedro Ramírez',   'Ventas',    55000, 36, 'Buenos Aires'),
        (5,  'Lucía Fernández', 'IT',        92000, 31, 'Buenos Aires'),
        (6,  'Juan Martínez',   'Finanzas',  48000, 45, 'Mendoza'),
        (7,  'Sofía Díaz',      'IT',        85000, 27, 'Córdoba'),
        (8,  'Diego Herrera',   'Ventas',    41000, 38, 'Rosario'),
        (9,  'Elena Ruiz',      'Finanzas',  67000, 52, 'Buenos Aires'),
        (10, 'Martín Castro',   'RRHH',      38000, 24, 'Mendoza'),
        (11, 'Valeria Suárez',  'Ventas',    71000, 33, 'Buenos Aires'),
        (12, 'Roberto Moreno',  'IT',        53000, 40, 'Córdoba');
""")

print('✅ Tabla empleados creada.')
ejecutar_query(conn_emp, 'SELECT * FROM empleados', 'Vista completa de empleados:')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Escribí una consulta que muestre nombre y salario
# de los empleados que ganan más de $50,000
# Orderalos de mayor a menor salario

mi_query = """
-- Escribí tu consulta aquí
"""

# ejecutar_query(conn_emp, mi_query, 'Empleados con salario > $50,000:')

<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
SELECT nombre, salario
FROM empleados
WHERE salario > 50000
ORDER BY salario DESC;
```

**Bonus:** ¿Podés también filtrar solo los de IT que ganan más de $50,000? Usá `AND`.

</details>

In [ ]:
# ✅ SOLUCIÓN

# --- Consulta básica ---
ejecutar_query(conn_emp, """
    SELECT nombre, salario
    FROM   empleados
    WHERE  salario > 50000
    ORDER  BY salario DESC
""", '1. Empleados con salario > $50,000 (ordenado):')

# --- Con múltiples condiciones ---
ejecutar_query(conn_emp, """
    SELECT nombre, departamento, salario
    FROM   empleados
    WHERE  salario > 50000
      AND  departamento = 'IT'
    ORDER  BY salario DESC
""", '2. Empleados de IT con salario > $50,000:')

# --- Con BETWEEN, LIKE e IN ---
ejecutar_query(conn_emp, """
    SELECT nombre, departamento, salario, ciudad
    FROM   empleados
    WHERE  salario BETWEEN 50000 AND 80000
      AND  departamento IN ('Ventas', 'IT')
    ORDER  BY departamento, salario DESC
""", '3. Ventas o IT con salario entre $50K y $80K:')

# --- Estadísticas por departamento ---
ejecutar_query(conn_emp, """
    SELECT
        departamento,
        COUNT(*)          AS cantidad,
        ROUND(AVG(salario), 0) AS salario_promedio,
        MIN(salario)      AS salario_minimo,
        MAX(salario)      AS salario_maximo
    FROM   empleados
    GROUP  BY departamento
    ORDER  BY salario_promedio DESC
""", '4. Estadísticas salariales por departamento:')

---
## 📦 Ejercicio 2 — JOIN: productos y categorías

**Enunciado:** Diseñá una consulta que muestre el nombre de los productos y sus categorías correspondientes (tablas separadas relacionadas por clave foránea).

### 📖 Explicación conceptual

**JOIN** combina filas de dos tablas cuando existe una relación entre ellas.

```
productos                    categorias
──────────────────────       ──────────────────
id | nombre | cat_id    →   id | nombre
1  | Laptop  |   2           1  | Accesorios
2  | Mouse   |   1           2  | Electrónica
```

| Tipo de JOIN | Devuelve |
|--------------|----------|
| `INNER JOIN` | Solo filas que coinciden en AMBAS tablas |
| `LEFT JOIN`  | TODAS las filas de la izquierda + coincidencias de la derecha |
| `RIGHT JOIN` | TODAS las filas de la derecha + coincidencias de la izquierda |
| `FULL JOIN`  | TODAS las filas de ambas tablas |

```sql
SELECT p.nombre, c.nombre AS categoria
FROM   productos p             -- alias 'p' para abreviar
JOIN   categorias c ON p.categoria_id = c.id;
```

In [ ]:
# 🔧 CREAMOS LA BASE DE DATOS DE PRODUCTOS
conn_prod = sqlite3.connect(':memory:')

conn_prod.executescript("""
    CREATE TABLE categorias (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL
    );

    CREATE TABLE productos (
        id           INTEGER PRIMARY KEY,
        nombre       TEXT    NOT NULL,
        precio       REAL    NOT NULL,
        stock        INTEGER DEFAULT 0,
        categoria_id INTEGER REFERENCES categorias(id)
    );

    INSERT INTO categorias VALUES
        (1, 'Electrónica'),
        (2, 'Accesorios'),
        (3, 'Audio'),
        (4, 'Almacenamiento');

    INSERT INTO productos VALUES
        (1,  'Laptop Pro',        1200.00, 15,   1),
        (2,  'Mouse Inalámbrico',   25.00, 80,   2),
        (3,  'Auriculares BT',      85.00, 40,   3),
        (4,  'Teclado Mecánico',   110.00, 30,   2),
        (5,  'Monitor 27"',        350.00, 12,   1),
        (6,  'Webcam HD',           60.00, 25,   1),
        (7,  'SSD 1TB',            120.00, 50,   4),
        (8,  'Hub USB',             18.00, 60,   2),
        (9,  'Parlante BT',         45.00, 35,   3),
        (10, 'Pendrive 64GB',        8.00, 100,  4),
        (11, 'Producto sin cat.',   20.00, 10,  NULL);
""")

print('✅ Tablas productos y categorias creadas.')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Consultá el nombre del producto y su categoría
# ¿Qué pasa con el producto que no tiene categoría?
# Probá INNER JOIN y LEFT JOIN y observá la diferencia

mi_query = """
-- Escribí tu consulta JOIN aquí
"""

# ejecutar_query(conn_prod, mi_query, 'Productos con sus categorías:')

<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
-- INNER JOIN: solo productos CON categoría
SELECT p.nombre, c.nombre AS categoria
FROM   productos p
JOIN   categorias c ON p.categoria_id = c.id;

-- LEFT JOIN: TODOS los productos (los sin categoría muestran NULL)
SELECT p.nombre, c.nombre AS categoria
FROM   productos p
LEFT JOIN categorias c ON p.categoria_id = c.id;
```
El producto sin categoría (`NULL` en `categoria_id`) aparece solo con `LEFT JOIN`.

</details>

In [ ]:
# ✅ SOLUCIÓN

# INNER JOIN — solo productos que tienen categoría asignada
ejecutar_query(conn_prod, """
    SELECT
        p.nombre          AS producto,
        c.nombre          AS categoria,
        p.precio,
        p.stock
    FROM  productos  p
    JOIN  categorias c ON p.categoria_id = c.id
    ORDER BY c.nombre, p.nombre
""", '1. INNER JOIN — productos con categoría:')

# LEFT JOIN — todos los productos, incluso sin categoría
ejecutar_query(conn_prod, """
    SELECT
        p.nombre                              AS producto,
        COALESCE(c.nombre, 'Sin categoría')   AS categoria,
        p.precio
    FROM  productos  p
    LEFT JOIN categorias c ON p.categoria_id = c.id
    ORDER BY categoria, p.nombre
""", '2. LEFT JOIN — todos los productos (COALESCE reemplaza NULL):')

# Conteo y valor total de stock por categoría
ejecutar_query(conn_prod, """
    SELECT
        c.nombre                          AS categoria,
        COUNT(p.id)                       AS cantidad_productos,
        ROUND(AVG(p.precio), 2)           AS precio_promedio,
        SUM(p.stock)                      AS stock_total,
        ROUND(SUM(p.precio * p.stock), 2) AS valor_inventario
    FROM  categorias c
    LEFT JOIN productos p ON p.categoria_id = c.id
    GROUP BY c.id, c.nombre
    ORDER BY valor_inventario DESC
""", '3. Resumen de inventario por categoría:')

---
## 📦 Ejercicio 3 — Crear tablas con relaciones: base de datos Biblioteca

**Enunciado:** Creá una base de datos `biblioteca` con las tablas `autores` y `libros`. Definí una relación uno-a-muchos: un autor puede tener varios libros.

### 📖 Explicación conceptual

Una **relación uno-a-muchos** (1:N) es la más común en bases de datos:

```
autores (1)  ─────────── (N) libros
Un autor     tiene muchos    libros
```

Se implementa con una **clave foránea** (`FOREIGN KEY`) en la tabla del lado "muchos":

```sql
CREATE TABLE libros (
    id        INTEGER PRIMARY KEY,
    titulo    TEXT NOT NULL,
    autor_id  INTEGER REFERENCES autores(id)  -- clave foránea
);
```

**Restricciones útiles:**

| Restricción | Significado |
|-------------|-------------|
| `PRIMARY KEY` | Identificador único, no nulo |
| `NOT NULL` | El campo no puede estar vacío |
| `UNIQUE` | No puede repetirse en la tabla |
| `DEFAULT valor` | Valor por defecto si no se especifica |
| `CHECK (condicion)` | Valida que el valor cumpla una condición |

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Creá una conexión SQLite en memoria
# 2. Creá la tabla 'autores' con: id, nombre, nacionalidad, año_nacimiento
# 3. Creá la tabla 'libros' con: id, titulo, año_publicacion, genero, autor_id (FK)
# 4. Insertá al menos 3 autores y 6 libros
# 5. Hacé una consulta que muestre el título del libro y el nombre del autor

conn_bib = sqlite3.connect(':memory:')

# Escribí tu código aquí


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
CREATE TABLE autores (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre        TEXT NOT NULL,
    nacionalidad  TEXT,
    anio_nac      INTEGER
);

CREATE TABLE libros (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo          TEXT NOT NULL,
    anio_publicacion INTEGER,
    genero          TEXT,
    autor_id        INTEGER NOT NULL REFERENCES autores(id)
);
```
Para insertar varios registros de una vez: `INSERT INTO tabla VALUES (...), (...), (...);`

</details>

In [ ]:
# ✅ SOLUCIÓN

conn_bib = sqlite3.connect(':memory:')

conn_bib.executescript("""
    -- Activar soporte de claves foráneas en SQLite
    PRAGMA foreign_keys = ON;

    -- Tabla padre: autores
    CREATE TABLE autores (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre       TEXT    NOT NULL,
        nacionalidad TEXT,
        anio_nac     INTEGER CHECK(anio_nac > 1000)  -- validación básica
    );

    -- Tabla hija: libros (tiene FK a autores)
    CREATE TABLE libros (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo           TEXT    NOT NULL,
        anio_publicacion INTEGER,
        genero           TEXT    DEFAULT 'Sin clasificar',
        paginas          INTEGER,
        autor_id         INTEGER NOT NULL REFERENCES autores(id)
    );

    -- Insertamos autores
    INSERT INTO autores (nombre, nacionalidad, anio_nac) VALUES
        ('Gabriel García Márquez', 'Colombiana', 1927),
        ('Jorge Luis Borges',      'Argentina',  1899),
        ('Isabel Allende',         'Chilena',    1942),
        ('Julio Cortázar',         'Argentina',  1914);

    -- Insertamos libros (autor_id referencia autores.id)
    INSERT INTO libros (titulo, anio_publicacion, genero, paginas, autor_id) VALUES
        ('Cien años de soledad',       1967, 'Realismo mágico', 471,  1),
        ('El amor en los tiempos del cólera', 1985, 'Novela', 348,    1),
        ('Ficciones',                  1944, 'Cuentos',         224,  2),
        ('El Aleph',                   1949, 'Cuentos',         203,  2),
        ('La casa de los espíritus',   1982, 'Novela',          448,  3),
        ('Eva Luna',                   1987, 'Novela',          320,  3),
        ('Rayuela',                    1963, 'Novela',          736,  4),
        ('Bestiario',                  1951, 'Cuentos',         168,  4);
""")

print('✅ Base de datos biblioteca creada.')

# Consulta con JOIN para ver libros + autores
ejecutar_query(conn_bib, """
    SELECT
        l.titulo,
        a.nombre          AS autor,
        a.nacionalidad,
        l.genero,
        l.anio_publicacion,
        l.paginas
    FROM  libros   l
    JOIN  autores  a ON l.autor_id = a.id
    ORDER BY a.nombre, l.anio_publicacion
""", 'Libros con sus autores:')

# Cuántos libros tiene cada autor
ejecutar_query(conn_bib, """
    SELECT
        a.nombre     AS autor,
        COUNT(l.id)  AS cantidad_libros,
        MIN(l.anio_publicacion) AS primer_libro,
        MAX(l.anio_publicacion) AS ultimo_libro
    FROM  autores a
    LEFT JOIN libros l ON l.autor_id = a.id
    GROUP BY a.id, a.nombre
    ORDER BY cantidad_libros DESC
""", 'Libros por autor:')

---
## 📦 Ejercicio 4 — Ampliar el modelo: tabla editoriales

**Enunciado:** Expandí la base de datos `biblioteca` agregando una tabla `editoriales`. Un libro pertenece a una editorial.

### 📖 Explicación conceptual

Ahora el modelo tiene **tres tablas relacionadas**:

```
autores (1) ──── (N) libros (N) ──── (1) editoriales
```

Para consultar las tres tablas a la vez encadenamos JOINs:
```sql
SELECT l.titulo, a.nombre, e.nombre
FROM   libros l
JOIN   autores    a ON l.autor_id     = a.id
JOIN   editoriales e ON l.editorial_id = e.id;
```

También podemos **agregar columnas** a tablas existentes con `ALTER TABLE`:
```sql
ALTER TABLE libros ADD COLUMN editorial_id INTEGER REFERENCES editoriales(id);
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Usando conn_bib (la conexión del ejercicio anterior):
# 1. Creá la tabla 'editoriales' con: id, nombre, pais, año_fundacion
# 2. Agregá la columna editorial_id a la tabla libros
# 3. Insertá editoriales y actualizá los libros para asignarles una
# 4. Consultá libros con autor Y editorial


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
-- Crear la tabla editorial
CREATE TABLE editoriales (
    id            INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL,
    pais          TEXT,
    anio_fund     INTEGER
);

-- Agregar columna a tabla existente
ALTER TABLE libros ADD COLUMN editorial_id INTEGER REFERENCES editoriales(id);

-- Asignar editorial a un libro
UPDATE libros SET editorial_id = 1 WHERE id = 1;
```

</details>

In [ ]:
# ✅ SOLUCIÓN

# Creamos la tabla editoriales y la vinculamos a libros
conn_bib.executescript("""
    CREATE TABLE editoriales (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre    TEXT    NOT NULL,
        pais      TEXT,
        anio_fund INTEGER
    );

    INSERT INTO editoriales (nombre, pais, anio_fund) VALUES
        ('Sudamericana',      'Argentina', 1939),
        ('Emecé',             'Argentina', 1939),
        ('Seix Barral',       'España',    1911),
        ('Plaza & Janés',     'España',    1959);

    -- Agregamos la columna FK a la tabla libros
    ALTER TABLE libros ADD COLUMN editorial_id INTEGER REFERENCES editoriales(id);

    -- Asignamos editoriales a los libros
    UPDATE libros SET editorial_id = 1 WHERE id IN (1, 2);  -- García Márquez → Sudamericana
    UPDATE libros SET editorial_id = 2 WHERE id IN (3, 4);  -- Borges → Emecé
    UPDATE libros SET editorial_id = 3 WHERE id IN (5, 6);  -- Allende → Seix Barral
    UPDATE libros SET editorial_id = 4 WHERE id IN (7, 8);  -- Cortázar → Plaza & Janés
""")

print('✅ Tabla editoriales creada y libros actualizados.')

# Consulta con triple JOIN
ejecutar_query(conn_bib, """
    SELECT
        l.titulo,
        a.nombre     AS autor,
        e.nombre     AS editorial,
        e.pais,
        l.anio_publicacion
    FROM  libros      l
    JOIN  autores     a ON l.autor_id     = a.id
    JOIN  editoriales e ON l.editorial_id = e.id
    ORDER BY a.nombre, l.anio_publicacion
""", 'Libros con autor y editorial (triple JOIN):')

# Cuántos libros tiene cada editorial
ejecutar_query(conn_bib, """
    SELECT
        e.nombre    AS editorial,
        e.pais,
        COUNT(l.id) AS cantidad_libros
    FROM  editoriales e
    LEFT JOIN libros l ON l.editorial_id = e.id
    GROUP BY e.id
    ORDER BY cantidad_libros DESC
""", 'Libros por editorial:')

---
## 📦 Ejercicio 5 — UPDATE: aumento de salario en Ventas

**Enunciado:** Actualizá el salario de todos los empleados del departamento de Ventas aumentándolo un 10%.

### 📖 Explicación conceptual

**UPDATE** modifica datos existentes. Su estructura es:
```sql
UPDATE tabla
SET    columna = nuevo_valor
WHERE  condicion;
```

⚠️ **¡Siempre incluí WHERE!** Sin WHERE, se actualizan TODAS las filas.

Para aumentar en un porcentaje: `salario = salario * 1.10` (multiplica por 1 + el porcentaje).

**Buena práctica:** antes de hacer el UPDATE, hacé un SELECT con la misma condición para verificar qué filas vas a modificar.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Primero consultá los empleados de Ventas y sus salarios actuales
# 2. Ejecutá el UPDATE para aumentar 10%
# 3. Verificá que el cambio se aplicó correctamente

# ejecutar_query(conn_emp, "SELECT ...", 'Antes del aumento:')
# ejecutar_query(conn_emp, "UPDATE ...", 'Actualización:')
# ejecutar_query(conn_emp, "SELECT ...", 'Después del aumento:')

<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
-- Verificar ANTES
SELECT nombre, salario FROM empleados WHERE departamento = 'Ventas';

-- Hacer el UPDATE
UPDATE empleados
SET    salario = salario * 1.10
WHERE  departamento = 'Ventas';

-- Verificar DESPUÉS
SELECT nombre, salario FROM empleados WHERE departamento = 'Ventas';
```

</details>

In [ ]:
# ✅ SOLUCIÓN

# Paso 1: Ver estado actual de Ventas
ejecutar_query(conn_emp, """
    SELECT nombre, departamento, salario
    FROM   empleados
    WHERE  departamento = 'Ventas'
    ORDER  BY nombre
""", 'ANTES del aumento — empleados de Ventas:')

# Paso 2: Guardar salarios originales para comparar después
df_antes = pd.read_sql_query(
    "SELECT nombre, salario FROM empleados WHERE departamento = 'Ventas'",
    conn_emp
)

# Paso 3: Ejecutar el UPDATE
ejecutar_query(conn_emp, """
    UPDATE empleados
    SET    salario = ROUND(salario * 1.10, 2)
    WHERE  departamento = 'Ventas'
""", 'Ejecutando aumento del 10%:')

# Paso 4: Verificar el resultado
ejecutar_query(conn_emp, """
    SELECT nombre, departamento, salario
    FROM   empleados
    WHERE  departamento = 'Ventas'
    ORDER  BY nombre
""", 'DESPUÉS del aumento — empleados de Ventas:')

# Paso 5: Mostrar comparación
df_despues = pd.read_sql_query(
    "SELECT nombre, salario FROM empleados WHERE departamento = 'Ventas'",
    conn_emp
)
df_comp = df_antes.rename(columns={'salario': 'salario_anterior'}).merge(
    df_despues.rename(columns={'salario': 'salario_nuevo'}), on='nombre'
)
df_comp['aumento'] = (df_comp['salario_nuevo'] - df_comp['salario_anterior']).round(2)
print('Comparación de salarios:')
print(df_comp.to_string(index=False))

---
## 📦 Ejercicio 6 — DELETE: eliminar productos baratos

**Enunciado:** Eliminá todos los productos con precio inferior a $10.

### 📖 Explicación conceptual

**DELETE** elimina filas de una tabla:
```sql
DELETE FROM tabla WHERE condicion;
```

⚠️ **Sin WHERE, borra TODA la tabla.** No hay Ctrl+Z en SQL puro.

**Buenas prácticas antes de DELETE:**
1. Hacé un `SELECT` con la misma condición para ver qué vas a borrar.
2. En producción, usá **transacciones** (`BEGIN` / `COMMIT` / `ROLLBACK`) para poder deshacer.

```sql
BEGIN;                             -- Inicio de transacción
DELETE FROM productos WHERE precio < 10;
-- Si algo salió mal:
ROLLBACK;                          -- Deshace los cambios
-- Si todo está bien:
COMMIT;                            -- Confirma los cambios
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Consultá los productos con precio < $10
# 2. Ejecutá el DELETE
# 3. Verificá cuántos productos quedaron


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
-- Verificar qué se va a borrar
SELECT * FROM productos WHERE precio < 10;

-- Borrar
DELETE FROM productos WHERE precio < 10;

-- Confirmar cuántos quedan
SELECT COUNT(*) AS total_restante FROM productos;
```

</details>

In [ ]:
# ✅ SOLUCIÓN

# Paso 1: Ver qué vamos a borrar
ejecutar_query(conn_prod, """
    SELECT id, nombre, precio
    FROM   productos
    WHERE  precio < 10
""", 'Productos a eliminar (precio < $10):')

total_antes = pd.read_sql_query('SELECT COUNT(*) AS total FROM productos', conn_prod)['total'][0]
print(f'Total de productos ANTES: {total_antes}')

# Paso 2: Borrar (usando transacción)
conn_prod.execute('BEGIN')
conn_prod.execute('DELETE FROM productos WHERE precio < 10')
conn_prod.execute('COMMIT')
print('\n✅ DELETE ejecutado con COMMIT.')

# Paso 3: Verificar resultado
total_despues = pd.read_sql_query('SELECT COUNT(*) AS total FROM productos', conn_prod)['total'][0]
print(f'Total de productos DESPUÉS: {total_despues}')
print(f'Productos eliminados: {total_antes - total_despues}')

ejecutar_query(conn_prod, """
    SELECT nombre, precio
    FROM   productos
    ORDER  BY precio
""", 'Productos restantes:')

---
## 📦 Ejercicio 7 — Consulta avanzada con JOIN múltiple y GROUP BY

**Enunciado:** Con tablas `clientes`, `pedidos` y `productos`, mostrá el nombre del cliente y el total gastado en todos sus pedidos.

### 📖 Explicación conceptual

**GROUP BY** agrupa filas con el mismo valor en una columna y permite aplicar funciones de agregación:

```sql
SELECT cliente, SUM(total) AS total_gastado
FROM   pedidos
GROUP  BY cliente
ORDER  BY total_gastado DESC;
```

**HAVING** filtra grupos (como WHERE pero para resultados agregados):
```sql
-- Clientes que gastaron más de $1000
HAVING SUM(total) > 1000
```

| Función | Descripción |
|---------|-------------|
| `COUNT(*)` | Cuenta filas |
| `SUM(col)` | Suma de valores |
| `AVG(col)` | Promedio |
| `MIN(col)` | Mínimo |
| `MAX(col)` | Máximo |
| `ROUND(val, n)` | Redondea a n decimales |

In [ ]:
# 🔧 CREAMOS LA BASE DE DATOS DE PEDIDOS
conn_ped = sqlite3.connect(':memory:')

conn_ped.executescript("""
    CREATE TABLE clientes (
        id     INTEGER PRIMARY KEY,
        nombre TEXT    NOT NULL,
        email  TEXT    UNIQUE,
        ciudad TEXT
    );

    CREATE TABLE productos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT    NOT NULL,
        precio REAL    NOT NULL
    );

    CREATE TABLE pedidos (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER REFERENCES clientes(id),
        producto_id INTEGER REFERENCES productos(id),
        cantidad    INTEGER NOT NULL,
        fecha       TEXT    NOT NULL
    );

    INSERT INTO clientes VALUES
        (1, 'Ana García',   'ana@mail.com',   'Buenos Aires'),
        (2, 'Carlos López', 'carlos@mail.com', 'Córdoba'),
        (3, 'María Torres', 'maria@mail.com',  'Rosario'),
        (4, 'Pedro Ruiz',   'pedro@mail.com',  'Mendoza');

    INSERT INTO productos VALUES
        (1, 'Laptop',   1200.00),
        (2, 'Mouse',      25.00),
        (3, 'Teclado',   110.00),
        (4, 'Monitor',   350.00),
        (5, 'Auriculares', 85.00);

    INSERT INTO pedidos (cliente_id, producto_id, cantidad, fecha) VALUES
        (1, 1, 1, '2024-01-10'),
        (1, 2, 2, '2024-01-15'),
        (1, 5, 1, '2024-02-01'),
        (2, 3, 1, '2024-01-20'),
        (2, 4, 2, '2024-02-05'),
        (3, 2, 3, '2024-01-25'),
        (3, 5, 2, '2024-02-10'),
        (4, 1, 2, '2024-02-15'),
        (4, 3, 1, '2024-02-20'),
        (1, 4, 1, '2024-03-01');
""")
print('✅ Base de datos de pedidos creada.')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Mostrá el nombre del cliente y el total gastado en todos sus pedidos
# Recordá que el total de un pedido = cantidad * precio del producto
# Ordenalo de mayor a menor gasto


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```sql
SELECT
    c.nombre,
    SUM(p_ped.cantidad * prod.precio) AS total_gastado
FROM   clientes    c
JOIN   pedidos     p_ped ON p_ped.cliente_id  = c.id
JOIN   productos   prod  ON p_ped.producto_id = prod.id
GROUP  BY c.id, c.nombre
ORDER  BY total_gastado DESC;
```
El triple JOIN conecta las tres tablas. El `GROUP BY` agrupa todos los pedidos de cada cliente.

</details>

In [ ]:
# ✅ SOLUCIÓN

# Consulta principal: total gastado por cliente
ejecutar_query(conn_ped, """
    SELECT
        c.nombre                                   AS cliente,
        c.ciudad,
        COUNT(ped.id)                              AS cant_pedidos,
        ROUND(SUM(ped.cantidad * prod.precio), 2)  AS total_gastado
    FROM  clientes  c
    JOIN  pedidos   ped  ON ped.cliente_id  = c.id
    JOIN  productos prod ON ped.producto_id = prod.id
    GROUP BY c.id, c.nombre, c.ciudad
    ORDER BY total_gastado DESC
""", '1. Total gastado por cliente:')

# Clientes que gastaron más de $1000 (HAVING)
ejecutar_query(conn_ped, """
    SELECT
        c.nombre                                   AS cliente,
        ROUND(SUM(ped.cantidad * prod.precio), 2)  AS total_gastado
    FROM  clientes  c
    JOIN  pedidos   ped  ON ped.cliente_id  = c.id
    JOIN  productos prod ON ped.producto_id = prod.id
    GROUP BY c.id
    HAVING SUM(ped.cantidad * prod.precio) > 1000
    ORDER BY total_gastado DESC
""", '2. Clientes VIP (gastaron más de $1,000) con HAVING:')

# Producto más vendido
ejecutar_query(conn_ped, """
    SELECT
        prod.nombre                  AS producto,
        SUM(ped.cantidad)            AS unidades_vendidas,
        ROUND(SUM(ped.cantidad * prod.precio), 2) AS ingresos
    FROM  pedidos   ped
    JOIN  productos prod ON ped.producto_id = prod.id
    GROUP BY prod.id, prod.nombre
    ORDER BY unidades_vendidas DESC
""", '3. Ventas por producto:')

---
## 📦 Ejercicio 8 — Subconsultas: estudiantes en más de 3 cursos

**Enunciado:** Usá una base de datos de `estudiantes` y `cursos` para mostrar los nombres de los estudiantes inscritos en más de 3 cursos.

### 📖 Explicación conceptual

Una **subconsulta** es una consulta dentro de otra consulta. Se puede usar en:
- `WHERE`: para filtrar según el resultado de otra consulta.
- `FROM`: como si fuera una tabla temporal.
- `SELECT`: para calcular valores adicionales.

```sql
-- Subconsulta en WHERE
SELECT nombre FROM estudiantes
WHERE id IN (
    SELECT estudiante_id
    FROM inscripciones
    GROUP BY estudiante_id
    HAVING COUNT(*) > 3
);
```

Esto es equivalente a usar `JOIN` + `HAVING`, pero la subconsulta puede ser más legible en ciertos casos.

In [ ]:
# 🔧 CREAMOS LA BASE DE DATOS DE ESTUDIANTES
conn_est = sqlite3.connect(':memory:')

conn_est.executescript("""
    CREATE TABLE estudiantes (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        edad   INTEGER,
        carrera TEXT
    );

    CREATE TABLE cursos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        creditos INTEGER
    );

    -- Tabla intermedia: relación muchos-a-muchos
    CREATE TABLE inscripciones (
        estudiante_id INTEGER REFERENCES estudiantes(id),
        curso_id      INTEGER REFERENCES cursos(id),
        fecha         TEXT,
        nota          REAL,
        PRIMARY KEY (estudiante_id, curso_id)
    );

    INSERT INTO estudiantes VALUES
        (1, 'Ana García',    22, 'Ingeniería'),
        (2, 'Carlos López',  20, 'Ciencias'),
        (3, 'María Torres',  23, 'Ingeniería'),
        (4, 'Pedro Ruiz',    21, 'Matemática'),
        (5, 'Lucía Díaz',    22, 'Ciencias');

    INSERT INTO cursos VALUES
        (1, 'Matemática I',     4),
        (2, 'Programación',     6),
        (3, 'Bases de Datos',   4),
        (4, 'Estadística',      4),
        (5, 'Redes',            3),
        (6, 'Algoritmos',       5);

    INSERT INTO inscripciones VALUES
        (1, 1, '2024-03-01', 8.5),
        (1, 2, '2024-03-01', 9.0),
        (1, 3, '2024-03-01', 7.5),
        (1, 4, '2024-03-01', 8.0),  -- Ana: 4 cursos
        (2, 1, '2024-03-01', 6.0),
        (2, 2, '2024-03-01', 7.0),  -- Carlos: 2 cursos
        (3, 1, '2024-03-01', 9.0),
        (3, 3, '2024-03-01', 8.5),
        (3, 5, '2024-03-01', 7.0),
        (3, 6, '2024-03-01', 9.5),  -- María: 4 cursos
        (4, 2, '2024-03-01', 5.5),
        (4, 4, '2024-03-01', 6.0),
        (4, 6, '2024-03-01', 7.0),  -- Pedro: 3 cursos
        (5, 1, '2024-03-01', 8.0),
        (5, 2, '2024-03-01', 9.5),
        (5, 3, '2024-03-01', 8.0),
        (5, 4, '2024-03-01', 7.5),
        (5, 5, '2024-03-01', 6.5); -- Lucía: 5 cursos
""")
print('✅ Base de datos de estudiantes creada.')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Mostrá los nombres de los estudiantes inscritos en MÁS DE 3 cursos
# Usá tanto la versión con subconsulta como la versión con JOIN + HAVING


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

**Opción A — con subconsulta:**
```sql
SELECT nombre FROM estudiantes
WHERE id IN (
    SELECT estudiante_id FROM inscripciones
    GROUP BY estudiante_id HAVING COUNT(*) > 3
);
```
**Opción B — con JOIN + HAVING:**
```sql
SELECT e.nombre, COUNT(*) AS cant_cursos
FROM estudiantes e JOIN inscripciones i ON i.estudiante_id = e.id
GROUP BY e.id HAVING COUNT(*) > 3;
```

</details>

In [ ]:
# ✅ SOLUCIÓN

# Opción A: Subconsulta en WHERE
ejecutar_query(conn_est, """
    SELECT nombre, carrera
    FROM   estudiantes
    WHERE  id IN (
        SELECT estudiante_id
        FROM   inscripciones
        GROUP  BY estudiante_id
        HAVING COUNT(*) > 3
    )
""", '1. Subconsulta — Estudiantes con más de 3 cursos:')

# Opción B: JOIN + GROUP BY + HAVING (más detalle)
ejecutar_query(conn_est, """
    SELECT
        e.nombre,
        e.carrera,
        COUNT(i.curso_id)      AS cant_cursos,
        ROUND(AVG(i.nota), 2)  AS promedio_notas,
        SUM(c.creditos)        AS creditos_totales
    FROM  estudiantes  e
    JOIN  inscripciones i ON i.estudiante_id = e.id
    JOIN  cursos        c ON i.curso_id       = c.id
    GROUP BY e.id, e.nombre, e.carrera
    HAVING COUNT(i.curso_id) > 3
    ORDER BY cant_cursos DESC
""", '2. JOIN + HAVING — Estudiantes con más de 3 cursos (detallado):')

# Resumen general de todos los estudiantes
ejecutar_query(conn_est, """
    SELECT
        e.nombre,
        COUNT(i.curso_id)      AS cant_cursos,
        ROUND(AVG(i.nota), 2)  AS promedio,
        CASE
            WHEN COUNT(i.curso_id) > 3 THEN '⚠️ Carga alta'
            WHEN COUNT(i.curso_id) = 3 THEN '✅ Normal'
            ELSE '📉 Baja carga'
        END AS estado
    FROM  estudiantes  e
    LEFT JOIN inscripciones i ON i.estudiante_id = e.id
    GROUP BY e.id, e.nombre
    ORDER BY cant_cursos DESC
""", '3. Resumen de todos los estudiantes con CASE:')

---
## 📦 Ejercicio 9 — Optimización con índices

**Enunciado:** Tomá una consulta que involucre múltiples tablas, creá índices para mejorar su rendimiento y medí el tiempo de ejecución antes y después.

### 📖 Explicación conceptual

Un **índice** es como el índice de un libro: en vez de leer todo para encontrar algo, vas directo a la página correcta.

```sql
-- Crear un índice en la columna salario
CREATE INDEX idx_salario ON empleados(salario);

-- Índice compuesto (para consultas que filtran por varias columnas)
CREATE INDEX idx_depto_sal ON empleados(departamento, salario);

-- Ver qué índices existen
SELECT * FROM sqlite_master WHERE type = 'index';
```

**¿Cuándo usar índices?**
- Columnas usadas frecuentemente en `WHERE`, `JOIN` y `ORDER BY`.
- Tablas grandes (miles de filas o más).

**¿Cuándo NO usarlos?**
- Tablas pequeñas (el índice puede ser más lento que scan completo).
- Columnas con pocos valores distintos (ej: sexo: M/F).
- Tablas con muchos INSERT/UPDATE (el índice se actualiza también).

In [ ]:
# 🔧 CREAMOS UNA TABLA GRANDE PARA MEDIR EL IMPACTO DEL ÍNDICE
import time

conn_big = sqlite3.connect(':memory:')
conn_big.execute("""
    CREATE TABLE ventas_grandes (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER,
        producto    TEXT,
        monto       REAL,
        fecha       TEXT,
        region      TEXT
    )
""")

import random
random.seed(42)
regiones   = ['Norte', 'Sur', 'Este', 'Oeste', 'Centro']
productos_ = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares']

N = 100_000  # 100 mil filas para notar diferencia
datos = [
    (i, random.randint(1, 1000), random.choice(productos_),
     round(random.uniform(10, 2000), 2),
     f'2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}',
     random.choice(regiones))
    for i in range(1, N+1)
]
conn_big.executemany('INSERT INTO ventas_grandes VALUES (?,?,?,?,?,?)', datos)
conn_big.commit()
print(f'✅ Tabla ventas_grandes creada con {N:,} filas.')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Medí el tiempo de una consulta SIN índice (filtrando por cliente_id y región)
# 2. Creá un índice sobre esas columnas
# 3. Medí el tiempo de la misma consulta CON índice
# 4. Comparás los tiempos


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
import time

query = "SELECT * FROM ventas_grandes WHERE cliente_id = 42 AND region = 'Norte'"

# Sin índice
t0 = time.perf_counter()
pd.read_sql_query(query, conn_big)
t1 = time.perf_counter()
print(f'Sin índice: {(t1-t0)*1000:.2f} ms')

# Crear índice
conn_big.execute('CREATE INDEX idx_cli_reg ON ventas_grandes(cliente_id, region)')

# Con índice
t0 = time.perf_counter()
pd.read_sql_query(query, conn_big)
t1 = time.perf_counter()
print(f'Con índice: {(t1-t0)*1000:.2f} ms')
```

</details>

In [ ]:
# ✅ SOLUCIÓN

query_filtro = """
    SELECT cliente_id, producto, SUM(monto) AS total
    FROM   ventas_grandes
    WHERE  cliente_id < 50
      AND  region = 'Norte'
    GROUP  BY cliente_id, producto
    ORDER  BY total DESC
"""

# --- SIN índice ---
repeticiones = 5
tiempos_sin = []
for _ in range(repeticiones):
    t0 = time.perf_counter()
    pd.read_sql_query(query_filtro, conn_big)
    tiempos_sin.append((time.perf_counter() - t0) * 1000)

media_sin = sum(tiempos_sin) / len(tiempos_sin)
print(f'⏱️  SIN índice — promedio de {repeticiones} ejecuciones: {media_sin:.2f} ms')

# --- Crear índices ---
conn_big.execute('CREATE INDEX idx_cli     ON ventas_grandes(cliente_id)')
conn_big.execute('CREATE INDEX idx_reg     ON ventas_grandes(region)')
conn_big.execute('CREATE INDEX idx_cli_reg ON ventas_grandes(cliente_id, region)')
conn_big.commit()
print('\n✅ Índices creados.')

# Ver los índices creados
indices = pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type='index' AND tbl_name='ventas_grandes'",
    conn_big
)
print('Índices en la tabla:')
print(indices.to_string(index=False))

# --- CON índice ---
tiempos_con = []
for _ in range(repeticiones):
    t0 = time.perf_counter()
    pd.read_sql_query(query_filtro, conn_big)
    tiempos_con.append((time.perf_counter() - t0) * 1000)

media_con = sum(tiempos_con) / len(tiempos_con)
print(f'\n⏱️  CON índice  — promedio de {repeticiones} ejecuciones: {media_con:.2f} ms')

mejora = ((media_sin - media_con) / media_sin) * 100
factor = media_sin / media_con if media_con > 0 else float('inf')
print(f'\n🚀 Mejora: {mejora:.1f}% más rápido ({factor:.1f}x)')

# Analizar el plan de consulta con EXPLAIN
print('\n📋 Plan de consulta (EXPLAIN QUERY PLAN):')
plan = pd.read_sql_query(f'EXPLAIN QUERY PLAN {query_filtro}', conn_big)
print(plan[['detail']].to_string(index=False))

---
## 📦 Ejercicio 10 — Subconsulta compleja con optimización

**Enunciado:** Diseñá una consulta SQL compleja que use una subconsulta para recuperar información específica. Aplicá una estrategia de optimización.

### 📖 Explicación conceptual

Las **subconsultas correlacionadas** referencian a la consulta exterior. Son poderosas pero pueden ser lentas:

```sql
-- Subconsulta correlacionada: para CADA empleado, busca su departamento
SELECT nombre,
       (SELECT MAX(salario) FROM empleados e2
        WHERE e2.departamento = e1.departamento) AS max_salario_depto
FROM   empleados e1;
```

**CTE (Common Table Expression)** — una forma más legible de escribir subconsultas:
```sql
WITH maximos AS (
    SELECT departamento, MAX(salario) AS max_sal
    FROM empleados GROUP BY departamento
)
SELECT e.nombre, e.salario, m.max_sal
FROM   empleados e
JOIN   maximos   m ON e.departamento = m.departamento;
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Usando conn_emp:
# 1. Escribí una consulta que muestre cada empleado junto con
#    el salario máximo de su departamento y cuánto gana por debajo del máximo
# 2. Primero con una subconsulta, luego con un CTE (WITH)
# 3. ¿Cuál te parece más legible?


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

**Con subconsulta:**
```sql
SELECT
    nombre, departamento, salario,
    (SELECT MAX(salario) FROM empleados e2
     WHERE e2.departamento = e1.departamento) AS max_depto,
    (SELECT MAX(salario) FROM empleados e2
     WHERE e2.departamento = e1.departamento) - salario AS diferencia
FROM empleados e1;
```
**Con CTE:**
```sql
WITH maximos AS (
    SELECT departamento, MAX(salario) AS max_sal
    FROM empleados GROUP BY departamento
)
SELECT e.nombre, e.departamento, e.salario,
       m.max_sal, m.max_sal - e.salario AS diferencia
FROM   empleados e JOIN maximos m ON e.departamento = m.departamento;
```

</details>

In [ ]:
# ✅ SOLUCIÓN

# Versión con subconsulta correlacionada
ejecutar_query(conn_emp, """
    SELECT
        e1.nombre,
        e1.departamento,
        e1.salario,
        (
            SELECT MAX(e2.salario)
            FROM   empleados e2
            WHERE  e2.departamento = e1.departamento
        )                            AS max_salario_depto,
        ROUND(
            (
                SELECT MAX(e2.salario)
                FROM   empleados e2
                WHERE  e2.departamento = e1.departamento
            ) - e1.salario, 2
        )                            AS diferencia_al_maximo
    FROM  empleados e1
    ORDER BY e1.departamento, e1.salario DESC
""", '1. Subconsulta correlacionada — salario vs máximo del depto:')

# Versión con CTE (más legible y eficiente: calcula el máximo una sola vez)
ejecutar_query(conn_emp, """
    WITH estadisticas_depto AS (
        SELECT
            departamento,
            MAX(salario)  AS max_sal,
            MIN(salario)  AS min_sal,
            ROUND(AVG(salario), 2) AS prom_sal,
            COUNT(*)      AS cant_empleados
        FROM  empleados
        GROUP BY departamento
    )
    SELECT
        e.nombre,
        e.departamento,
        e.salario,
        d.max_sal                          AS max_depto,
        d.prom_sal                         AS prom_depto,
        ROUND(e.salario - d.prom_sal, 2)   AS vs_promedio,
        CASE
            WHEN e.salario >= d.max_sal THEN '🏆 Top'
            WHEN e.salario >= d.prom_sal THEN '✅ Sobre promedio'
            ELSE '📉 Bajo promedio'
        END AS posicion
    FROM  empleados          e
    JOIN  estadisticas_depto d ON e.departamento = d.departamento
    ORDER BY e.departamento, e.salario DESC
""", '2. CTE — análisis completo de salarios por departamento:')

---
## 🏆 Resumen del notebook

| Ejercicio | Tema | Conceptos clave |
|-----------|------|-----------------|
| 1 | SELECT + WHERE | Filtros, operadores, ORDER BY, GROUP BY |
| 2 | JOIN | INNER JOIN, LEFT JOIN, alias, COALESCE |
| 3 | CREATE TABLE | PRIMARY KEY, FK, NOT NULL, AUTOINCREMENT |
| 4 | ALTER TABLE | Agregar columnas, relaciones múltiples |
| 5 | UPDATE | Modificar filas, transacciones |
| 6 | DELETE | Borrar filas, BEGIN/COMMIT/ROLLBACK |
| 7 | GROUP BY + HAVING | Agregaciones, múltiples JOINs |
| 8 | Subconsultas | WHERE IN (...), CASE WHEN |
| 9 | Índices | CREATE INDEX, EXPLAIN QUERY PLAN, medición |
| 10 | CTE + Optimización | WITH, subconsultas correlacionadas |

### 💪 Desafíos extra
1. **Ej. 2:** Agregá una tabla `proveedores` y vinculala a `productos`. Consultá producto + categoría + proveedor.
2. **Ej. 7:** Calculá el ticket promedio por cliente y mostrá solo los que están sobre el promedio general.
3. **Ej. 8:** Encontrá los estudiantes que están en TODOS los cursos de su carrera.
4. **Ej. 9:** Medí el impacto del índice en un `ORDER BY` sobre la columna `monto`.
5. **Ej. 10:** Reescribí la subconsulta del ej. 8 usando un CTE y compará la legibilidad.